# SMS SPAM DETECTION (NAIVE BAYES)

# Data Collection
- Data is collection from Datasets section on Kaggle

# Data Loading
- Faced Error while loading "'utf-8' codec can't decode bytes in position 606-607: invalid continuation byte"
- Meaning : Python is trying to read a file as UTF-8, but the file is actually saved in a different encoding
  
- Solution : Specified a different encoding (latin-1)

In [1]:
import pandas as pd
df = pd.read_csv("spam.csv",encoding="latin-1")

# Initial Exploration

- After Exploring data i noticed that dataset has 5 columns and 5572 rows
- Out of 5 , 2 rows are of my concern
- After checking missing values , we have missing values on unnecessary rows which willbe dropped
- i will name columns for my ease as well
- Check imbalance of ham vs spam: IR=747 / 4825 ​ = 6.46 (Moderate imbalance-accuracy maybe misleading) 
- Average Message length = 15.50 (short text , perfect for naive bayes)

In [2]:
df.rename(columns={'v1':'label'},inplace=True)
df.rename(columns={'v2':'message'},inplace=True)
df = df[['label','message']]

In [3]:
# df.shape
# df.head()
# df.isna().sum()
vc = df['label'].value_counts()
vc_avg = df['label'].value_counts().max() / df['label'].value_counts().min()
print(vc)
print(vc_avg)



label
ham     4825
spam     747
Name: count, dtype: int64
6.459170013386881


In [4]:
 message_length = []

 for text in df['message']:
    word = text.split()
    count= len(word)
    message_length.append(count)

df['message_length']=message_length


df['message_length'].mean()

np.float64(15.494436468054559)

# Data Cleaning 
- i have dropped unnecessary columns which also have missing values that can disturb our model
- Renamed columns names for easy understanding of data
- lower case text
- punctuatiion noise
- common words like is,this and
- different form of same words(stemming)
- naive bayes assumes clean word counts
- Tokenization

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
import re
import nltk


In [7]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to C:\Users\Adeel
[nltk_data]     Rana\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Adeel
[nltk_data]     Rana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Adeel
[nltk_data]     Rana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Importing Libraries

In [8]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

Text Cleaning Function

In [9]:
def clean_text(text):
 #Lowercase
 text = text.lower()

#Remove punctuation and special text

 text = re.sub(r'[^a-z\s]', '', text)

#Tokenization
 
 tokens = word_tokenize(text)

#Remove stop words

 cleaned_token=[]
 for word in tokens:
  if word not in stop_words:
    cleaned_token.append(word)

 #Join token back to sentance 

 cleaned_text = ' '.join(cleaned_token)

 return cleaned_text



In [10]:
#New columns of cleaned text
df['clean_message'] = df['message'].apply(clean_text)

In [11]:
#Checking null values and  adopting safety measures
df['clean_message'].isna().sum()
df = df[df['clean_message'].str.strip() !='']

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB



In [13]:
df.head()

,label,message,message_length,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,11,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goes usf lives around though


# Split Data
- Stratify = y  to preserve class distribution of label(y) 

In [15]:
X = df['clean_message']
y = df['label']

X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


# Pipeline implementation
- Prevents data leakage
- clean and readable
- Easy hyperparameter tuning later

In [30]:
pipeline = Pipeline([
    ("vectorizer",CountVectorizer()),
    ('model',MultinomialNB(alpha=0.5))
])

pipeline.fit(X_train,y_train)

y_pred = pipeline.predict(X_test)

# MODEL EVALUATION 
- accuracy can be misleading due to imbalanced data 
- precision high means there is still chance of improvement in risk of false positives(user experience POV)
- recall singals that model is detected 88% right spams sms
- f1 balances both

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix , classification_report

accuracy = accuracy_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred)
# recall = recall_score(y_test, y_pred)
# f1_value = f1_score(y_test, y_pred)
Confusion_M = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test,y_pred)
print("Accuracy",accuracy)
print("Classification Report",cr)
print("confusion",Confusion_M)

Accuracy 0.9748653500897666
Classification Report               precision    recall  f1-score   support

         ham       0.98      0.99      0.99       964
        spam       0.93      0.88      0.90       150

    accuracy                           0.97      1114
   macro avg       0.96      0.93      0.94      1114
weighted avg       0.97      0.97      0.97      1114

confusion [[954  10]
 [ 18 132]]


# Hyperparameter tuning
- Alpha = 0.5 in MultinomialNB , 1 can underfit 
- Used this handle zero probabilities and improve generalization.

# Before Tuning
Accuracy 0.9757630161579892
- Classification Report               
- ''''''''''''''```````````````precision'''recall``````````f1-score`````   support

         ham       0.98      0.99      0.99       964
        spam       0.94      0.87      0.91       150

    accuracy                           0.98      1114
   macro avg       0.96      0.93      0.95      1114
weighted avg       0.98      0.98      0.98      1114

confusion [[956   8]
 [ 19 131]]

# After Tuning
Accuracy 0.9757630161579892
- Classification Report               
- ''''''''''''''```````````````precision'''recall``````````f1-score`````   support

         ham       0.98      0.99      0.99       964
        spam       0.93      0.88      0.90       150

    accuracy                           0.97      1114
   macro avg       0.96      0.93      0.94      1114
weighted avg       0.97      0.97      0.97      1114

confusion [[954  10]
 [ 18 132]]

# Error Analysis 
- Checking False Negatives messages to see the error cause
- found out that words with no meaning are confusing the model that increased False Negatives 

In [33]:
results = pd.DataFrame({
    'message': X_test,
    'actual': y_test,
    'predicted': y_pred
})

F_N = results[
    (results['actual']=='spam')&
    (results['predicted']=='ham')
]

F_N.head()

,message,actual,predicted
730,email alertfrom jeri stewartsize kbsubject low...,spam,ham
3062,hi babe jordan r u im home abroad lonely text ...,spam,ham
3979,ringtoneking,spam,ham
1429,sale arsenal dartboard good condition doubles ...,spam,ham
2246,hi ya babe x u goten bout scammers getting sma...,spam,ham
